# LeetCode 1120 — Maximum Average Subtree bằng Python

**Link đề:** https://leetcode.com/problems/maximum-average-subtree/
**Độ khó:** Easy (theo danh mục của repo) · **Chủ đề:** Tree, Depth-First Search, Binary Tree

> Ghi chú: LeetCode hiện ghi độ khó bài này là **Medium**; notebook vẫn xếp vào `DSA/Easy` theo danh mục được giao.

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def maximumAverageSubtree(self, root: Optional[TreeNode]) -> float:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân** (binary tree), **cây con** (subtree) và cách tự dựng cây từ danh sách level-order để chạy notebook độc lập.
2. Nắm **đệ quy** trên cây: mỗi nút hỏi con trái, con phải rồi tự tổng hợp.
3. Thấy vì sao cách "tính lại từng cây con" tốn `O(n²)`, và kỹ thuật **DFS hậu thứ tự (post-order) trả về nhiều giá trị** (`(tổng, số nút)`) đưa về `O(n)`.
4. Viết bản **không đệ quy** bằng stack để tránh giới hạn độ sâu đệ quy của Python.
5. Trace, chứng minh tính đúng, Time/Space, bộ test đối chiếu, biến thể và bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — cây con có trung bình lớn nhất

Cho `root` của một cây nhị phân. Hãy trả về **giá trị trung bình lớn nhất** trong tất cả các **cây con** của cây.

- **Cây con** của cây là một nút bất kỳ **cùng với tất cả hậu duệ** (con, cháu, …) của nó.
- **Trung bình** của một cây = tổng giá trị các nút / số nút.
- Đáp án sai lệch trong phạm vi `10^-5` so với đáp án chuẩn được chấp nhận.

**Ví dụ 1:** `root = [5,6,1]`

```text
    5
   / \
  6   1
```

| Cây con gốc tại | Các nút | Trung bình |
|---|---|---|
| 5 | 5, 6, 1 | 12 / 3 = 4.0 |
| 6 | 6 | 6 / 1 = 6.0 |
| 1 | 1 | 1 / 1 = 1.0 |

→ Output `6.00000`.

**Ví dụ 2:** `root = [0,null,1]` → Output `1.00000`.

**Constraints (theo đề gốc):** số nút trong khoảng `[1, 10^4]`, `0 <= Node.val <= 10^5`.

In [1]:
# Tính tay ví dụ 1
subtrees = {"gốc 5": [5, 6, 1], "gốc 6": [6], "gốc 1": [1]}
for name, vals in subtrees.items():
    print(f"{name}: tổng={sum(vals)}, số nút={len(vals)}, trung bình={sum(vals) / len(vals)}")

gốc 5: tổng=12, số nút=3, trung bình=4.0
gốc 6: tổng=6, số nút=1, trung bình=6.0
gốc 1: tổng=1, số nút=1, trung bình=1.0


## 2. Nền tảng: cây nhị phân trong Python

### 2.1. Lớp `TreeNode`

Mỗi nút là một **đối tượng** có 3 thuộc tính: `val` (giá trị), `left` và `right` (tham chiếu tới nút con, hoặc `None` nếu không có). LeetCode định nghĩa sẵn lớp này; trong notebook ta tự định nghĩa để chạy độc lập.

`class` khai báo kiểu dữ liệu mới; `__init__` là hàm khởi tạo chạy khi tạo đối tượng `TreeNode(5)`; `self` là chính đối tượng đang được tạo.

In [2]:
from typing import Optional, List, Tuple

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

# Dựng tay ví dụ 1
root = TreeNode(5, TreeNode(6), TreeNode(1))
print(root, root.left, root.right, root.left.left)

TreeNode(5) TreeNode(6) TreeNode(1) None


### 2.2. Dựng cây từ danh sách level-order (định dạng LeetCode)

LeetCode mô tả cây bằng list duyệt **theo tầng** (level-order), `None` (`null`) nghĩa là vị trí trống: `[5,6,1]`, `[0,null,1]`.

Thuật toán dựng: dùng **hàng đợi** (`collections.deque`) chứa các nút chờ được gắn con. Lấy nút đầu hàng đợi, gắn lần lượt con trái và con phải từ list, nút con mới (khác `None`) được đưa vào cuối hàng đợi.

`deque` là hàng đợi hai đầu: `append` thêm vào cuối, `popleft` lấy ra từ đầu, cả hai `O(1)`.

In [3]:
from collections import deque

def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def print_tree(node: Optional[TreeNode], prefix: str = "", label: str = "root") -> None:
    # In cây xoay ngang: mỗi tầng thụt vào 4 khoảng trắng
    if node is None:
        return
    print(prefix + f"{label}: {node.val}")
    print_tree(node.left, prefix + "    ", "L")
    print_tree(node.right, prefix + "    ", "R")

print_tree(build_tree([5, 6, 1]))
print("---")
print_tree(build_tree([0, None, 1]))
print("---")
print_tree(build_tree([1, 2, 3, 4, None, 5, 6]))

root: 5
    L: 6
    R: 1
---
root: 0
    R: 1
---
root: 1
    L: 2
        L: 4
    R: 3
        L: 5
        R: 6


### 2.3. Đệ quy trên cây — mẫu tư duy

Một hàm đệ quy trên cây luôn có:
1. **Trường hợp cơ sở** (base case): nút `None` → trả giá trị "rỗng" (ví dụ tổng 0).
2. **Bước đệ quy:** gọi hàm cho `node.left`, `node.right`, rồi **kết hợp** với `node.val`.

Ví dụ: tổng mọi nút và đếm số nút.

In [4]:
def tree_sum(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return node.val + tree_sum(node.left) + tree_sum(node.right)

def tree_count(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + tree_count(node.left) + tree_count(node.right)

t = build_tree([1, 2, 3, 4, None, 5, 6])
print("tổng =", tree_sum(t), "| số nút =", tree_count(t))

tổng = 21 | số nút = 6


### 2.4. Hàm trả về nhiều giá trị (tuple) và phép chia `/`

Hàm Python có thể `return a, b` — thực chất trả **tuple** `(a, b)`; người gọi tách bằng `s, c = f(...)`.

Phép chia `/` luôn cho số thực (`float`): `12 / 3 = 4.0`. Đề yêu cầu trả `float`, nên dùng `/` chứ **không** dùng `//` (chia nguyên).

In [5]:
def sum_and_count(values):
    return sum(values), len(values)

s, c = sum_and_count([5, 6, 1])
print("s =", s, "| c =", c, "| tuple:", sum_and_count([5, 6, 1]))
print("12 / 3 =", 12 / 3, "| 7 / 2 =", 7 / 2, "| 7 // 2 =", 7 // 2)

s = 12 | c = 3 | tuple: (12, 3)
12 / 3 = 4.0 | 7 / 2 = 3.5 | 7 // 2 = 3


## 3. Cách 1 — Brute Force: với mỗi nút, tính lại tổng và số nút của cây con

Ý tưởng trực giác: duyệt mọi nút `v`; với mỗi `v`, gọi `tree_sum(v)` và `tree_count(v)` để có trung bình; lấy max.

In [6]:
def all_nodes(node: Optional[TreeNode]) -> List[TreeNode]:
    if node is None:
        return []
    return [node] + all_nodes(node.left) + all_nodes(node.right)

def max_avg_brute(root: Optional[TreeNode]) -> float:
    best = 0.0
    for v in all_nodes(root):
        best = max(best, tree_sum(v) / tree_count(v))
    return best

print(max_avg_brute(build_tree([5, 6, 1])))      # 6.0
print(max_avg_brute(build_tree([0, None, 1])))   # 1.0

6.0
1.0


`best = 0.0` an toàn vì mọi giá trị nút `>= 0` nên mọi trung bình `>= 0`.

### Nút thắt: tính lặp lại

Mỗi nút ở độ sâu `d` bị cộng lại **`d + 1` lần** (một lần cho mỗi tổ tiên của nó, kể cả chính nó). Với cây "lệch" (mỗi nút chỉ có một con — giống danh sách liên kết), tổng công việc là `1 + 2 + ... + n = O(n²)`. Với `n = 10^4` là ~`5·10^7` lần thăm nút — chậm trong Python.

Đo số lần thăm nút:

In [7]:
visits = 0
def tree_sum_counting(node):
    global visits
    if node is None:
        return 0
    visits += 1
    return node.val + tree_sum_counting(node.left) + tree_sum_counting(node.right)

def build_skewed(n):                      # cây lệch phải: 1 -> 2 -> ... -> n
    root = cur = TreeNode(1)
    for v in range(2, n + 1):
        cur.right = TreeNode(v)
        cur = cur.right
    return root

for n in [10, 100, 500]:
    visits = 0
    t = build_skewed(n)
    for v in all_nodes(t):
        tree_sum_counting(v)
    print(f"n={n:>3}: số lần thăm nút để tính các tổng = {visits:>6} = n(n+1)/2 = {n * (n + 1) // 2}")

n= 10: số lần thăm nút để tính các tổng =     55 = n(n+1)/2 = 55
n=100: số lần thăm nút để tính các tổng =   5050 = n(n+1)/2 = 5050
n=500: số lần thăm nút để tính các tổng = 125250 = n(n+1)/2 = 125250


`global visits` cho phép hàm **sửa** biến `visits` ở ngoài hàm (mặc định, gán trong hàm tạo biến cục bộ mới). Chỉ dùng cho thí nghiệm đếm như thế này.

## 4. Insight — thông tin của cây con cha được **ghép** từ hai cây con

Gọi `S(v)` là tổng, `C(v)` là số nút của cây con gốc `v`:

```text
S(v) = v.val + S(v.left) + S(v.right)
C(v) = 1     + C(v.left) + C(v.right)
S(None) = 0, C(None) = 0
```

Nếu mỗi lời gọi trả về **cả** `(S, C)`, cha chỉ việc cộng — không cần tính lại cây con. Thứ tự tự nhiên: xử lý **con trước, cha sau** → **duyệt hậu thứ tự** (post-order: trái → phải → gốc). Trong lúc trả về, ta cập nhật biến `best` toàn cục với `S(v) / C(v)`.

Mỗi nút được thăm đúng **1 lần** → `O(n)`.

In [8]:
# Tính tay theo công thức, từ lá lên gốc, cho ví dụ 1
S6, C6 = 6, 1
S1, C1 = 1, 1
S5, C5 = 5 + S6 + S1, 1 + C6 + C1
print("gốc 6:", S6 / C6, "| gốc 1:", S1 / C1, "| gốc 5:", S5, "/", C5, "=", S5 / C5)

gốc 6: 6.0 | gốc 1: 1.0 | gốc 5: 12 / 3 = 4.0


## 5. Cách 2 — DFS hậu thứ tự trả về `(tổng, số nút)` (lời giải chính)

Mảnh ghép:
- Hàm lồng `dfs(node)` trả `(sum, count)`.
- Base case: `None` → `(0, 0)`.
- Gọi `dfs` cho hai con, cộng lại, cập nhật `best`, trả về.
- `nonlocal best` cho phép hàm lồng **sửa** biến `best` của hàm bao ngoài.

In [9]:
def outer():
    counter = 0
    def inner():
        nonlocal counter        # không có dòng này -> UnboundLocalError
        counter += 1
    inner(); inner()
    return counter

print("nonlocal demo:", outer())

nonlocal demo: 2


In [10]:
def max_avg_dfs(root: Optional[TreeNode]) -> float:
    best = 0.0

    def dfs(node: Optional[TreeNode]) -> Tuple[int, int]:
        nonlocal best
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left)
        rs, rc = dfs(node.right)
        s = node.val + ls + rs
        c = 1 + lc + rc
        best = max(best, s / c)
        return s, c

    dfs(root)
    return best

print(max_avg_dfs(build_tree([5, 6, 1])))       # 6.0
print(max_avg_dfs(build_tree([0, None, 1])))    # 1.0
print(max_avg_dfs(build_tree([1, 2, 3, 4, None, 5, 6])))

6.0
1.0
6.0


## 6. TRACE từng bước

Cây `[1, 2, 3, 4, None, 5, 6]`:

```text
        1
      /   \
     2     3
    /     / \
   4     5   6
```

Thứ tự hậu thứ tự: `4, 2, 5, 6, 3, 1`.

| Nút | (ls, lc) | (rs, rc) | s | c | s/c | best sau |
|---:|---|---|---:|---:|---:|---:|
| 4 | (0,0) | (0,0) | 4 | 1 | 4.0 | 4.0 |
| 2 | (4,1) | (0,0) | 6 | 2 | 3.0 | 4.0 |
| 5 | (0,0) | (0,0) | 5 | 1 | 5.0 | 5.0 |
| 6 | (0,0) | (0,0) | 6 | 1 | 6.0 | 6.0 |
| 3 | (5,1) | (6,1) | 14 | 3 | 4.667 | 6.0 |
| 1 | (6,2) | (14,3) | 21 | 6 | 3.5 | 6.0 |

Đáp án `6.0` (lá 6).

In [11]:
def trace_max_avg(root: Optional[TreeNode]) -> float:
    best = 0.0

    def dfs(node, depth):
        nonlocal best
        if node is None:
            return 0, 0
        indent = "  " * depth
        print(f"{indent}vào nút {node.val}")
        ls, lc = dfs(node.left, depth + 1)
        rs, rc = dfs(node.right, depth + 1)
        s, c = node.val + ls + rs, 1 + lc + rc
        best = max(best, s / c)
        print(f"{indent}ra nút {node.val}: trái=({ls},{lc}) phải=({rs},{rc}) -> s={s}, c={c}, s/c={s / c:.3f}, best={best:.3f}")
        return s, c

    dfs(root, 0)
    print("Kết quả:", best)
    return best

_ = trace_max_avg(build_tree([1, 2, 3, 4, None, 5, 6]))

vào nút 1
  vào nút 2
    vào nút 4
    ra nút 4: trái=(0,0) phải=(0,0) -> s=4, c=1, s/c=4.000, best=4.000
  ra nút 2: trái=(4,1) phải=(0,0) -> s=6, c=2, s/c=3.000, best=4.000
  vào nút 3
    vào nút 5
    ra nút 5: trái=(0,0) phải=(0,0) -> s=5, c=1, s/c=5.000, best=5.000
    vào nút 6
    ra nút 6: trái=(0,0) phải=(0,0) -> s=6, c=1, s/c=6.000, best=6.000
  ra nút 3: trái=(5,1) phải=(6,1) -> s=14, c=3, s/c=4.667, best=6.000
ra nút 1: trái=(6,2) phải=(14,3) -> s=21, c=6, s/c=3.500, best=6.000
Kết quả: 6.0


Quan sát: dòng "ra nút" xuất hiện theo đúng thứ tự hậu thứ tự `4, 2, 5, 6, 3, 1` — con luôn "ra" trước cha, nên khi cha tính `s, c` thì số liệu của con đã sẵn sàng.

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def maximumAverageSubtree(self, root: Optional[TreeNode]) -> float:
        best = 0.0

        def dfs(node):
            nonlocal best
            if not node:
                return 0, 0
            ls, lc = dfs(node.left)
            rs, rc = dfs(node.right)
            s = node.val + ls + rs
            c = 1 + lc + rc
            best = max(best, s / c)
            return s, c

        dfs(root)
        return best

sol = Solution()
print(sol.maximumAverageSubtree(build_tree([5, 6, 1])))
print(sol.maximumAverageSubtree(build_tree([0, None, 1])))

6.0
1.0


## 8. Vấn đề độ sâu đệ quy — Cách 3: DFS hậu thứ tự bằng stack

Python giới hạn độ sâu đệ quy mặc định khoảng **1000** (`sys.getrecursionlimit()`). Cây lệch với `10^4` nút có độ sâu `10^4` → đệ quy có thể gây `RecursionError` trên máy cục bộ. (Môi trường chấm của LeetCode thường đã nâng giới hạn này, nên bản đệ quy vẫn được chấp nhận, nhưng hiểu vấn đề và biết cách khử đệ quy là kỹ năng quan trọng.)

In [13]:
import sys
print("recursion limit:", sys.getrecursionlimit())
deep = build_skewed(5000)
try:
    print(max_avg_dfs(deep))
except RecursionError as e:
    print("RecursionError:", str(e)[:60], "...")

recursion limit: 1000
RecursionError: maximum recursion depth exceeded ...


**Khử đệ quy:** dùng stack chứa cặp `(node, visited)`:
- Lần đầu gặp nút (`visited=False`): đẩy lại nó với `visited=True`, rồi đẩy hai con → con sẽ được xử lý **trước**.
- Lần thứ hai (`visited=True`): hai con đã xong, kết quả của chúng nằm trong dict `info[node] = (s, c)` → tính cho nút hiện tại.

Dict dùng **đối tượng nút** làm key được (mặc định mỗi đối tượng băm theo định danh).

In [14]:
def max_avg_iterative(root: Optional[TreeNode]) -> float:
    best = 0.0
    info = {None: (0, 0)}                 # nút None có (tổng, số nút) = (0, 0)
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if node is None:
            continue
        if not visited:
            stack.append((node, True))    # quay lại sau khi xong hai con
            stack.append((node.right, False))
            stack.append((node.left, False))
        else:
            ls, lc = info[node.left]
            rs, rc = info[node.right]
            s, c = node.val + ls + rs, 1 + lc + rc
            info[node] = (s, c)
            best = max(best, s / c)
    return best

print(max_avg_iterative(build_tree([5, 6, 1])))
print(max_avg_iterative(build_tree([1, 2, 3, 4, None, 5, 6])))
print("cây lệch 5000 nút:", max_avg_iterative(deep))

6.0
6.0
cây lệch 5000 nút: 5000.0


Cây lệch `1 → 2 → ... → 5000`: cây con có trung bình lớn nhất là **lá** `5000` (mọi cây con khác chứa các số nhỏ hơn) → kết quả `5000.0`. Bản code nộp không đệ quy:

In [15]:
class SolutionIterative:
    def maximumAverageSubtree(self, root: Optional[TreeNode]) -> float:
        best = 0.0
        info = {None: (0, 0)}
        stack = [(root, False)]
        while stack:
            node, visited = stack.pop()
            if node is None:
                continue
            if visited:
                ls, lc = info[node.left]
                rs, rc = info[node.right]
                s, c = node.val + ls + rs, 1 + lc + rc
                info[node] = (s, c)
                best = max(best, s / c)
            else:
                stack.extend([(node, True), (node.right, False), (node.left, False)])
        return best

print(SolutionIterative().maximumAverageSubtree(deep))

5000.0


## 9. Vì sao đúng? — Độ phức tạp

**Tính đúng:** chứng minh bằng quy nạp theo chiều cao cây: `dfs(v)` trả về đúng `(S(v), C(v))`.
- Cơ sở: `dfs(None) = (0, 0)` đúng theo định nghĩa.
- Bước: nếu `dfs` đúng với hai con thì `s = v.val + S(left) + S(right) = S(v)` và `c = 1 + C(left) + C(right) = C(v)`.

Mỗi nút `v` được gọi `dfs` đúng một lần và tại đó `best` được so với `S(v)/C(v)`. Vì **mọi** cây con đều có dạng "nút `v` + toàn bộ hậu duệ", nên `best` là max trên tất cả cây con. Bản stack tính đúng cùng các giá trị vì nút chỉ được xử lý (`visited=True`) sau khi cả hai con đã có trong `info`.

**Độ phức tạp** (`n` = số nút, `h` = chiều cao cây):

| Cách | Time | Space |
|---|---|---|
| 1. Brute Force | `O(n²)` worst (cây lệch), `O(n log n)` cây cân bằng | `O(h)` stack đệ quy + `O(n)` list nút |
| 2. DFS đệ quy | `O(n)` | `O(h)` stack đệ quy (`h` từ `log n` tới `n`) |
| 3. DFS stack | `O(n)` | `O(n)` cho `info` + stack |

Về số thực: tổng tối đa `10^4 · 10^5 = 10^9` vẫn là `int` chính xác; chỉ phép chia cuối mới sinh `float` (sai số rất nhỏ, nằm trong `10^-5`).

## 10. Bộ test — mẫu, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert abs(actual - expected) < 1e-5, (name, actual, expected)
    print(f"{name:<34} PASS {actual:.5f}")

cases = [
    ("mẫu 1", [5, 6, 1], 6.0),
    ("mẫu 2", [0, None, 1], 1.0),
    ("1 nút", [7], 7.0),
    ("toàn 0", [0, 0, 0], 0.0),
    ("cả cây là tốt nhất", [5, 5, 5], 5.0),
    ("hòa giữa cây con 3 nút và lá", [1, 10, 1, 10, 10], 10.0),
    ("cây con 2 nút > mọi lá", [0, 9, None, 8, None], 8.5),
    ("cây ví dụ trace", [1, 2, 3, 4, None, 5, 6], 6.0),
]
for name, values, exp in cases:
    check(name, Solution().maximumAverageSubtree(build_tree(values)), exp)

mẫu 1                              PASS 6.00000
mẫu 2                              PASS 1.00000
1 nút                              PASS 7.00000
toàn 0                             PASS 0.00000
cả cây là tốt nhất                 PASS 5.00000
hòa giữa cây con 3 nút và lá       PASS 10.00000
cây con 2 nút > mọi lá             PASS 8.50000
cây ví dụ trace                    PASS 6.00000


Kiểm thử đối chiếu với cây ngẫu nhiên (seed cố định): Brute Force làm oracle.

In [17]:
import random
rng = random.Random(1120)

def random_tree(n, max_val=20):
    if n == 0:
        return None
    nodes = [TreeNode(rng.randint(0, max_val)) for _ in range(n)]
    for i in range(1, n):                       # gắn nút i vào một nút trước đó còn chỗ trống
        while True:
            p = nodes[rng.randrange(i)]
            if p.left is None and rng.random() < 0.5:
                p.left = nodes[i]; break
            if p.right is None:
                p.right = nodes[i]; break
            if p.left is None:
                p.left = nodes[i]; break
    return nodes[0]

for _ in range(500):
    t = random_tree(rng.randint(1, 30))
    expected = max_avg_brute(t)
    for f in (max_avg_dfs, max_avg_iterative, Solution().maximumAverageSubtree,
              SolutionIterative().maximumAverageSubtree):
        assert abs(f(t) - expected) < 1e-9
print("500 cây ngẫu nhiên: 4 cách khớp Brute Force")

500 cây ngẫu nhiên: 4 cách khớp Brute Force


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ xét trung bình tại các lá hoặc chỉ tại gốc

In [18]:
def wrong_only_root(root):
    return tree_sum(root) / tree_count(root)      # SAI: chỉ 1 cây con

t = build_tree([5, 6, 1])
print("sai:", wrong_only_root(t), "| đúng:", max_avg_dfs(t))

sai: 4.0 | đúng: 6.0


### Lỗi 2: Trung bình của cha = trung bình của hai con (sai toán học)

In [19]:
def wrong_average_of_averages(root):
    best = 0.0
    def dfs(node):
        nonlocal best
        if node is None:
            return None
        avgs = [a for a in (dfs(node.left), dfs(node.right)) if a is not None]
        avg = (node.val + sum(avgs)) / (1 + len(avgs))   # SAI: bỏ qua trọng số số nút
        best = max(best, avg)
        return avg
    dfs(root)
    return best

t = build_tree([0, 9, None, 8, None, 1])     # nhánh trái nhiều nút
print("sai:", wrong_average_of_averages(t), "| đúng:", max_avg_dfs(t))
t2 = build_tree([10, 0, 0, 0, 0])
print("sai:", wrong_average_of_averages(t2), "| đúng:", max_avg_dfs(t2))

sai: 6.75 | đúng: 6.0
sai: 3.3333333333333335 | đúng: 2.0


**Vì sao sai:** "trung bình của các trung bình" chỉ đúng khi các nhóm cùng kích thước. Với cây `[10, 0, 0, 0, 0]`, cả cây có 5 nút, tổng 10 → trung bình `2.0`; nhưng cách sai tính `(10 + 0 + 0)/3 ≈ 3.33` cho gốc (coi nhánh trái 3 nút như 1 nút) và trả `3.33` trong khi đáp án đúng là `2.0`. **Sửa:** truyền lên **tổng và số nút**, chỉ chia ở mỗi nút để so sánh.

### Lỗi 3: Dùng `//` thay vì `/`

In [20]:
print("7 // 2 =", 7 // 2, "(mất phần thập phân) | 7 / 2 =", 7 / 2)

7 // 2 = 3 (mất phần thập phân) | 7 / 2 = 3.5


### Lỗi 4: Quên `nonlocal` khi gán biến của hàm bao ngoài

In [21]:
def wrong_no_nonlocal(root):
    best = 0.0
    def dfs(node):
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left); rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        best = max(best, s / c)          # SAI: Python coi best là biến cục bộ
        return s, c
    dfs(root)
    return best

try:
    wrong_no_nonlocal(build_tree([5, 6, 1]))
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'best' where it is not associated with a value


**Sửa:** thêm `nonlocal best`, hoặc dùng list một phần tử `best = [0.0]` và gán `best[0] = ...` (sửa nội dung, không gán lại tên), hoặc biến thuộc tính `self.best`.

## 12. Mẹo nhận dạng pattern

- Đề hỏi một đại lượng trên **mọi cây con** (tổng, trung bình, kích thước, có hợp lệ không, …) → **DFS hậu thứ tự**, mỗi nút trả về "bản tóm tắt" cây con của nó.
- Đại lượng không tự cộng được (trung bình, tỉ lệ) → trả về các thành phần **cộng được** (tổng, số lượng) rồi mới tính.
- Đáp án là max/min trên mọi cây con → biến `best` dùng chung (`nonlocal`), cập nhật lúc "ra" khỏi nút.
- Cây có thể sâu `10^4` → cân nhắc stack thay cho đệ quy trong Python.
- Bài cùng họ: #508 Most Frequent Subtree Sum, #2265 Count Nodes Equal to Average of Subtree, #1373 Maximum Sum BST in Binary Tree, #543 Diameter of Binary Tree, #563 Binary Tree Tilt.

## 13. Biến thể

### 13.1. #2265 — Đếm số nút có giá trị bằng trung bình (làm tròn xuống) của cây con của nó

Cùng khung `(tổng, số nút)`, chỉ đổi phần "cập nhật kết quả".

In [22]:
def count_nodes_equal_to_avg(root: Optional[TreeNode]) -> int:
    count = 0
    def dfs(node):
        nonlocal count
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left); rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        if node.val == s // c:
            count += 1
        return s, c
    dfs(root)
    return count

print(count_nodes_equal_to_avg(build_tree([4, 8, 5, 0, 1, None, 6])))   # 5
print(count_nodes_equal_to_avg(build_tree([1])))                        # 1

5
1


### 13.2. Trả về **nút gốc** của cây con có trung bình lớn nhất (thay vì giá trị)

Lưu thêm `best_node` mỗi khi `best` được cải thiện (dùng `>` để giữ nút gặp trước khi hòa).

In [23]:
def best_subtree_root(root: Optional[TreeNode]) -> Optional[TreeNode]:
    best, best_node = -1.0, None
    def dfs(node):
        nonlocal best, best_node
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left); rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        if s / c > best:
            best, best_node = s / c, node
        return s, c
    dfs(root)
    return best_node

print(best_subtree_root(build_tree([1, 2, 3, 4, None, 5, 6])))

TreeNode(6)


## 14. Bài tập (tăng dần)

**Bài 1 — Tính tay:** với cây `[3, 1, 8, None, None, 7, 9]`, lập bảng hậu thứ tự `(s, c, s/c)` và cho đáp án.

**Bài 2 — Dự đoán output:** với cây lệch `1 → 2 → 3` (mỗi nút là con phải của nút trước), đáp án là bao nhiêu? Tại sao trên cây lệch tăng dần đáp án luôn là lá?

**Bài 3 — TODO:** hoàn thiện DFS trả `(tổng, số nút)`.

In [24]:
def exercise_max_avg(root: Optional[TreeNode]) -> float:
    best = 0.0
    def dfs(node):
        # TODO 1: nonlocal best
        # TODO 2: base case None -> (0, 0)
        # TODO 3: gọi dfs hai con, tính s, c, cập nhật best, return s, c
        pass
    dfs(root)
    return best

# assert abs(exercise_max_avg(build_tree([5, 6, 1])) - 6.0) < 1e-9

**Bài 4 — Biến thể:** trả về **tổng lớn nhất** của một cây con (Maximum Subtree Sum). Giá trị nút có thể âm.

In [25]:
def exercise_max_subtree_sum(root: Optional[TreeNode]) -> int:
    # TODO
    pass

# assert exercise_max_subtree_sum(build_tree([-5, 3, -1, 2, 4])) == 9

**Bài 5 — Kích thước:** trả về trung bình lớn nhất trong các cây con có **ít nhất `k` nút**; nếu không có cây con nào đủ `k` nút, trả `-1.0`.

In [26]:
def exercise_max_avg_at_least_k(root: Optional[TreeNode], k: int) -> float:
    # TODO
    pass

# assert abs(exercise_max_avg_at_least_k(build_tree([5, 6, 1]), 2) - 4.0) < 1e-9

**Bài 6 — Mini challenge:** viết bản **không đệ quy** của Bài 4 bằng stack `(node, visited)`.

In [27]:
def exercise_max_subtree_sum_iterative(root: Optional[TreeNode]) -> int:
    # TODO
    pass

# assert exercise_max_subtree_sum_iterative(build_tree([-5, 3, -1, 2, 4])) == 9

### 14.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Thứ tự: 1, 7, 9, 8, 3 | nút 8: s = 24, c = 3 | lá 9 cho 9.0 |
| 2 | Liệt kê 3 cây con | `(1+2+3)/3`, `(2+3)/2`, `3/1` | `3.0`; thêm nút nhỏ hơn chỉ kéo trung bình xuống |
| 3 | Làm như mục 5 | `ls, lc = dfs(node.left)` | `best = max(best, s / c)` |
| 4 | Chỉ cần trả tổng | `best` khởi tạo `-inf` | `s = val + dfs(l) + dfs(r)` |
| 5 | Vẫn trả `(s, c)` | chỉ cập nhật khi `c >= k` | `best = -1.0` ban đầu |
| 6 | Dùng `info = {None: 0}` | xử lý nút khi `visited` | `info[node] = node.val + info[l] + info[r]` |

## 15. Đáp án tham khảo

- **Bài 1:** hậu thứ tự `1 (1,1,1.0)`, `7 (7,1,7.0)`, `9 (9,1,9.0)`, `8 (24,3,8.0)`, `3 (28,5,5.6)` → đáp án **9.0**.
- **Bài 2:** cây con `{1,2,3}` → 2.0, `{2,3}` → 2.5, `{3}` → 3.0 → đáp án **3.0**. Trên cây lệch tăng dần, cây con của nút nào cũng gồm nút đó và các số lớn hơn; lá là số lớn nhất, đứng riêng một mình nên có trung bình lớn nhất.

In [28]:
def answer_max_avg(root):
    best = 0.0
    def dfs(node):
        nonlocal best
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left); rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        best = max(best, s / c)
        return s, c
    dfs(root)
    return best

def answer_max_subtree_sum(root):
    best = float("-inf")
    def dfs(node):
        nonlocal best
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        best = max(best, s)
        return s
    dfs(root)
    return best

def answer_max_avg_at_least_k(root, k):
    best = -1.0
    def dfs(node):
        nonlocal best
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left); rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        if c >= k:
            best = max(best, s / c)
        return s, c
    dfs(root)
    return best

def answer_max_subtree_sum_iterative(root):
    best = float("-inf")
    info = {None: 0}
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if node is None:
            continue
        if visited:
            info[node] = node.val + info[node.left] + info[node.right]
            best = max(best, info[node])
        else:
            stack.extend([(node, True), (node.right, False), (node.left, False)])
    return best

assert abs(answer_max_avg(build_tree([3, 1, 8, None, None, 7, 9])) - 9.0) < 1e-9
assert abs(answer_max_avg(build_skewed(3)) - 3.0) < 1e-9
assert abs(answer_max_avg(build_tree([5, 6, 1])) - 6.0) < 1e-9
assert answer_max_subtree_sum(build_tree([-5, 3, -1, 2, 4])) == 9
assert abs(answer_max_avg_at_least_k(build_tree([5, 6, 1]), 2) - 4.0) < 1e-9
assert answer_max_avg_at_least_k(build_tree([5, 6, 1]), 4) == -1.0
assert answer_max_subtree_sum_iterative(build_tree([-5, 3, -1, 2, 4])) == 9
for _ in range(300):
    t = random_tree(rng.randint(1, 25), max_val=10)
    assert answer_max_subtree_sum(t) == answer_max_subtree_sum_iterative(t)
    assert abs(answer_max_avg(t) - max_avg_brute(t)) < 1e-9
    assert abs(answer_max_avg_at_least_k(t, 1) - max_avg_brute(t)) < 1e-9
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 16. Template nhớ nhanh + checklist submit

```python
def solve(root):
    best = 0.0
    def dfs(node):                 # trả về "bản tóm tắt" của cây con
        nonlocal best
        if not node:
            return 0, 0            # (tổng, số nút) của cây rỗng
        ls, lc = dfs(node.left)
        rs, rc = dfs(node.right)
        s, c = node.val + ls + rs, 1 + lc + rc
        best = max(best, s / c)    # cập nhật đáp án tại mỗi nút
        return s, c
    dfs(root)
    return best
```

**Checklist:**
- [ ] Trả về **tổng và số nút**, không trả trung bình lên cha?
- [ ] Dùng `/` (float) khi tính trung bình?
- [ ] Có `nonlocal best` (hoặc `self.best`)?
- [ ] Base case `None → (0, 0)` và không chia cho 0 (chỉ chia khi `c >= 1`)?
- [ ] Đã nghĩ tới cây lệch rất sâu (giới hạn đệ quy)?